# SOV-Transcribe-1 — request, then weights

Salesforce SOV-Transcribe-1 is a 26.6B-parameter speech model (4.6B active). This notebook builds the chat request the model card describes. It does **not** download the checkpoint unless you set `RUN_MODEL = True` on a machine with CUDA.

The field guide, with the benchmark table and the sources, is the repository README. This file is the runnable edge of that guide.

Model card: <https://huggingface.co/salesforce-sovereign-intelligence/SOV-Transcribe-1>


## The request

One user turn. The instruction is text. The recording is one audio part. Thinking stays off. Decoding is greedy (`temperature` 0). Audio is 16 kHz mono, and the published tests stop at 30 minutes.


In [ ]:
from pathlib import Path

PROMPTS = {
    "verbatim": "Transcribe the audio verbatim, outputting only spoken words in sequence.",
    "keywords": "This audio may contain the following words or phrases: {keywords}. Transcribe the audio into text.",
    "language_auto": "Transcribe the audio verbatim in the language's standard writing system, using punctuation and capitalization. Output only the transcript.",
    "language_hint": "Transcribe the {language} audio into text, ensuring all punctuation marks are included.",
}

MODEL_ID = "salesforce-sovereign-intelligence/SOV-Transcribe-1"

def messages(audio_path, prompt):
    path = Path(audio_path).expanduser().resolve()
    return [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "audio_url", "audio_url": {"url": path.as_uri()}},
        ],
    }]

sample = messages("meeting.wav", PROMPTS["keywords"].format(keywords="Greyling, invoice"))
print(MODEL_ID)
for part in sample[0]["content"]:
    print(part["type"] + ":", part.get("text") or part["audio_url"]["url"])


## Inference stays off

Set `RUN_MODEL` to `True` only after you have read the remote code in the model repository and you have a CUDA GPU large enough for the BF16 weights. Salesforce measured serving on a 96 GB RTX PRO 6000. The plugin path installs vLLM 0.32.0.

On a laptop this cell prints the reason and stops.


In [ ]:
from pathlib import Path

RUN_MODEL = False
AUDIO = "meeting.wav"

def cuda_ready():
    try:
        import torch
    except ImportError:
        return False
    return torch.cuda.is_available()

if not RUN_MODEL:
    print("RUN_MODEL is False. No weights downloaded.")
    print("Flip it on a CUDA machine, point AUDIO at a real file, then run this cell again.")
elif not Path(AUDIO).expanduser().is_file():
    raise SystemExit(f"No audio file at {AUDIO}")
elif not cuda_ready():
    raise SystemExit("CUDA is not available. The published checkpoint is a 26.6B BF16 model.")
else:
    import librosa
    from transformers import AutoModelForMultimodalLM, AutoProcessor

    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, device_map="auto"
    )
    wave, _ = librosa.load(AUDIO, sr=16000, mono=True)
    prompt = PROMPTS["verbatim"]
    inputs = processor.apply_transcription_request([wave], prompt=prompt).to(model.device, dtype=model.dtype)
    outputs = model.generate(**inputs, max_new_tokens=8192)
    transcript = processor.batch_decode(
        outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    print(transcript[0])


## The serving path Salesforce recommends

vLLM, not this notebook, is the path in the model card. The plugin lives in the weight repository and pins vLLM 0.32.0.

```bash
hf download salesforce-sovereign-intelligence/SOV-Transcribe-1 \
  --include "vllm_plugin/*" --local-dir SOV-Transcribe-1
pip install ./SOV-Transcribe-1/vllm_plugin
vllm serve salesforce-sovereign-intelligence/SOV-Transcribe-1 --max-model-len 32768
```

The endpoint allows 600 seconds of audio until `VLLM_MAX_AUDIO_DECODE_DURATION_S=1800`. Pass `-F temperature=0`. Pass `-F prompt=...` when you want a line other than the verbatim default.
